In [24]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import zipfile

from skimage.feature import hog, local_binary_pattern
from skimage import exposure

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [25]:
zip_file = "hymenoptera_data.zip"

In [26]:
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall("dataset3")

dataset_path = "dataset3"

In [27]:
image_paths = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            image_paths.append(os.path.join(root, file))

print("Total images found:", len(image_paths))

Total images found: 397


In [28]:
valid_paths = []
corrupted = []

for path in image_paths:
    img = cv2.imread(path)

    if img is None:
        corrupted.append(path)
    else:
        valid_paths.append(path)

print("Valid Images:", len(valid_paths))
print("Corrupted Images:", len(corrupted))

Valid Images: 397
Corrupted Images: 0


In [29]:
def get_label(path):
    ignore = ["train", "test", "valid", "validation", "images", "dataset"]

    parts = os.path.normpath(path).split(os.sep)

    for folder in reversed(parts[:-1]):
        if folder.lower() not in ignore:
            return folder

    return "unknown"

In [ ]:
for idx, path in enumerate(valid_paths[:3]):
    print(f"\nProcessing Image {idx+1}: {path}")

    image = cv2.imread(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    resized = cv2.resize(image, (256, 256))
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)

    rotated = cv2.warpAffine(
        resized,
        cv2.getRotationMatrix2D((128, 128), 45, 1.0),
        (256, 256)
    )

    flipped = cv2.flip(resized, 1)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    _, threshold = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    edges = cv2.Canny(gray, 100, 200)

    orb = cv2.ORB_create()
    keypoints, _ = orb.detectAndCompute(gray, None)

    orb_image = cv2.drawKeypoints(
        resized,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    hog_features, hog_image = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True
    )

    hog_image = exposure.rescale_intensity(hog_image)

    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')

    plt.figure(figsize=(18, 14))

    images = [
        (resized, "Original"),
        (gray, "Grayscale"),
        (rotated, "Rotated"),
        (flipped, "Flipped"),
        (blurred, "Blurred"),
        (threshold, "Threshold"),
        (edges, "Canny Edges"),
        (orb_image, "ORB Keypoints"),
        (hog_image, "HOG Features"),
        (lbp, "LBP Texture")
    ]

    for i, (img, title) in enumerate(images, 1):
        plt.subplot(3, 4, i)
        cmap = 'gray' if len(img.shape) == 2 else None
        plt.imshow(img, cmap=cmap)
        plt.title(title)
        plt.axis("off")

    plt.subplot(3, 4, 11)
    plt.hist(gray.ravel(), bins=256)
    plt.title("Color Histogram")

    plt.tight_layout()
    plt.show()

In [31]:
def extract_features(path):
    image = cv2.imread(path)

    if image is None:
        raise ValueError("Corrupted image")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (128, 128))

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False
    )

    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    lbp_hist = lbp_hist.astype(float)
    lbp_hist /= (lbp_hist.sum() + 1e-7)

    color_features = []

    for i in range(3):
        hist = cv2.calcHist([image], [i], None, [32], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        color_features.extend(hist)

    return np.hstack([hog_features, lbp_hist, color_features])

In [ ]:
X = []
y = []

for path in valid_paths:
    try:
        X.append(extract_features(path))
        y.append(get_label(path))
    except:
        pass

X = np.array(X)
y = np.array(y)

print("Detected Classes:", np.unique(y))

if len(np.unique(y)) != 2:
    raise ValueError("Binary classification requires exactly 2 classes")


encoder = LabelEncoder()
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
models = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(name, "Accuracy:", acc)


best_model_name = max(results, key=results.get)
predictions = models[best_model_name].predict(X_test)

print("Best Model:", best_model_name)
print(classification_report(y_test, predictions))

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')

for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(results.keys()), list(results.values()), marker='o')
plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.show()